# Smart Twin Prediction

This notebook combines the two trained machine-learning models:

- **XGBoost classifier** → failure prediction and failure probability
- **LightGBM regressor** → Remaining Useful Life (RUL) prediction

The Smart Twin layer combines these predictions into:

- Failure risk
- RUL status
- Overall machine status
- Maintenance recommendation
- Machine health score

**No model training happens in this notebook.**


In [1]:
# ============================================
# STEP 1: IMPORTS & SETUP
# ============================================

import os
import joblib
import pandas as pd
import numpy as np

print("Environment ready.")


Environment ready.


In [2]:
# ============================================
# STEP 2: LOAD TRAINED MODELS
# ============================================

failure_model_path = "../models/failure_prediction_model.pkl"
rul_model_path = "../models/rul_prediction_model.pkl"

failure_model = joblib.load(
    failure_model_path
)

rul_model = joblib.load(
    rul_model_path
)

print("Models loaded successfully!")

print("\nFailure Model:")
print(type(failure_model))

print("\nRUL Model:")
print(type(rul_model))


Models loaded successfully!

Failure Model:
<class 'sklearn.pipeline.Pipeline'>

RUL Model:
<class 'sklearn.pipeline.Pipeline'>


In [3]:
# ============================================
# STEP 3: MODEL FEATURE DEFINITIONS
# ============================================

failure_features = [
    "Cycle",
    "Temperature_C",
    "Vibration_mm_s",
    "Pressure_bar",
    "Rotational_Speed_RPM",
    "Load_Fraction",
    "Torque_Nm",
    "Current_A",
    "Acoustic_dB",
    "Wear_Level",
    "Machine_Type"
]

rul_features = [
    "Cycle",
    "Temperature_C",
    "Vibration_mm_s",
    "Pressure_bar",
    "Rotational_Speed_RPM",
    "Load_Fraction",
    "Torque_Nm",
    "Current_A",
    "Acoustic_dB",
    "Wear_Level",
    "Health_Index",
    "Anomaly_Score",
    "Machine_Type"
]

print("Failure model features:", len(failure_features))
print("RUL model features:", len(rul_features))


Failure model features: 11
RUL model features: 13


In [4]:
# ============================================
# STEP 4: LOAD MACHINE DATA
# ============================================

data_path = "../data/all_data/all_machines_combined.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Shape: (1930, 17)


,Machine_ID,Machine_Type,Failure_Mode,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Rotational_Speed_RPM,Load_Fraction,Torque_Nm,Current_A,Acoustic_dB,Wear_Level,Health_Index,Anomaly_Score,Failure,RUL_Cycles
0,CNC_001,CNC_Mill,Bearing,1,68.987,0.2351,2.117,1518.64,0.6513,46.734,9.213,56.022,0.0000,100.000,0.1342,0,259
1,CNC_001,CNC_Mill,Bearing,2,69.431,0.1894,2.126,1517.67,0.5961,46.603,9.480,55.789,0.0000,99.961,0.2990,0,258
2,CNC_001,CNC_Mill,Bearing,3,68.779,0.2385,2.119,1520.83,0.6336,46.313,9.612,54.954,0.0022,99.760,0.2269,0,257
3,CNC_001,CNC_Mill,Bearing,4,68.926,0.2047,2.110,1531.69,0.6898,48.403,9.907,54.567,0.0000,99.884,0.0469,0,256
4,CNC_001,CNC_Mill,Bearing,5,68.704,0.2086,2.079,1516.70,0.5900,48.188,9.080,54.424,0.0006,99.802,0.2841,0,255


In [5]:
# ============================================
# STEP 5: VALIDATE REQUIRED FEATURES
# ============================================

required_features = sorted(
    set(failure_features + rul_features)
)

missing_features = [
    feature
    for feature in required_features
    if feature not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing required features: {missing_features}"
    )

print("All required features are available.")
print("Total required features:", len(required_features))


All required features are available.
Total required features: 13


In [6]:
# ============================================
# STEP 6: MAINTENANCE RECOMMENDATION LOGIC
# ============================================

def get_maintenance_recommendation(
    failure_risk,
    predicted_rul
):

    if failure_risk == "HIGH" or predicted_rul <= 20:
        return "IMMEDIATE MAINTENANCE"

    elif failure_risk == "MEDIUM" or predicted_rul <= 50:
        return "SCHEDULE INSPECTION"

    else:
        return "NORMAL OPERATION"


In [7]:
# ============================================
# STEP 7: MACHINE HEALTH SCORE
# ============================================

def calculate_health_score(
    failure_probability,
    predicted_rul
):

    # Failure component: 0 to 100
    failure_score = (
        1 - failure_probability
    ) * 100

    # RUL component capped at 100 cycles
    rul_score = min(
        predicted_rul,
        100
    )

    # Combined health score
    health_score = (
        0.6 * failure_score
        +
        0.4 * rul_score
    )

    return round(
        health_score,
        2
    )


In [8]:
# ============================================
# STEP 8: SMART TWIN PREDICTION FUNCTION
# ============================================

def predict_machine_state(machine_data):

    # Convert input into DataFrame
    machine_df = pd.DataFrame([machine_data])

    # ----------------------------------------
    # FAILURE PREDICTION
    # ----------------------------------------

    failure_input = machine_df[
        failure_features
    ]

    failure_prediction = failure_model.predict(
        failure_input
    )[0]

    failure_probability = failure_model.predict_proba(
        failure_input
    )[0][1]

    # ----------------------------------------
    # RUL PREDICTION
    # ----------------------------------------

    rul_input = machine_df[
        rul_features
    ]

    predicted_rul = rul_model.predict(
        rul_input
    )[0]

    # RUL cannot logically be negative
    predicted_rul = max(
        0,
        float(predicted_rul)
    )

    # ----------------------------------------
    # FAILURE RISK
    # ----------------------------------------

    if failure_probability >= 0.70:
        failure_risk = "HIGH"

    elif failure_probability >= 0.30:
        failure_risk = "MEDIUM"

    else:
        failure_risk = "LOW"

    # ----------------------------------------
    # RUL STATUS
    # ----------------------------------------

    if predicted_rul <= 20:
        rul_status = "CRITICAL"

    elif predicted_rul <= 50:
        rul_status = "WARNING"

    else:
        rul_status = "HEALTHY"

    # ----------------------------------------
    # OVERALL MACHINE STATUS
    # ----------------------------------------

    if (
        failure_risk == "HIGH"
        or rul_status == "CRITICAL"
    ):
        overall_status = "CRITICAL"

    elif (
        failure_risk == "MEDIUM"
        or rul_status == "WARNING"
    ):
        overall_status = "WARNING"

    else:
        overall_status = "HEALTHY"

    # ----------------------------------------
    # MAINTENANCE RECOMMENDATION
    # ----------------------------------------

    maintenance_action = get_maintenance_recommendation(
        failure_risk,
        predicted_rul
    )

    # ----------------------------------------
    # HEALTH SCORE
    # ----------------------------------------

    health_score = calculate_health_score(
        failure_probability,
        predicted_rul
    )

    # ----------------------------------------
    # FINAL SMART TWIN RESULT
    # ----------------------------------------

    return {
        "failure_prediction": int(
            failure_prediction
        ),

        "failure_probability": round(
            float(failure_probability),
            4
        ),

        "failure_risk": failure_risk,

        "predicted_rul_cycles": round(
            predicted_rul,
            2
        ),

        "rul_status": rul_status,

        "overall_status": overall_status,

        "maintenance_action": maintenance_action,

        "health_score": health_score
    }


In [9]:
# ============================================
# STEP 9: TEST ONE MACHINE
# ============================================

sample_machine = df.iloc[1050]

smart_twin_result = predict_machine_state(
    sample_machine.to_dict()
)

print("SMART TWIN RESULT")
print("=================")

for key, value in smart_twin_result.items():
    print(f"{key}: {value}")


SMART TWIN RESULT
failure_prediction: 0
failure_probability: 0.0004
failure_risk: LOW
predicted_rul_cycles: 55.29
rul_status: HEALTHY
overall_status: HEALTHY
maintenance_action: NORMAL OPERATION
health_score: 82.08999633789062


In [10]:
# ============================================
# STEP 10: BATCH SMART TWIN PREDICTION
# ============================================

all_results = []

for _, row in df.iterrows():

    result = predict_machine_state(
        row.to_dict()
    )

    all_results.append(result)

smart_twin_results = pd.DataFrame(
    all_results
)

print("Smart Twin predictions generated.")
print("Result shape:", smart_twin_results.shape)

display(
    smart_twin_results.head(10)
)


Smart Twin predictions generated.
Result shape: (1930, 8)


,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
0,0,0.0001,LOW,257.17,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
1,0,0.0001,LOW,258.95,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
2,0,0.0001,LOW,259.01,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
3,0,0.0001,LOW,256.91,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
4,0,0.0001,LOW,260.68,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
5,0,0.0001,LOW,254.38,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
6,0,0.0001,LOW,251.90,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
7,0,0.0001,LOW,251.83,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
8,0,0.0001,LOW,251.49,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
9,0,0.0001,LOW,250.00,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998


In [11]:
# ============================================
# STEP 11: COMPLETE SMART TWIN OUTPUT
# ============================================

machine_information = df[
    [
        "Machine_ID",
        "Machine_Type",
        "Cycle",
        "Temperature_C",
        "Vibration_mm_s",
        "Pressure_bar",
        "Wear_Level",
        "Health_Index",
        "Anomaly_Score"
    ]
].reset_index(drop=True)

smart_twin_output = pd.concat(
    [
        machine_information,
        smart_twin_results.reset_index(drop=True)
    ],
    axis=1
)

print(
    "Complete Smart Twin output shape:",
    smart_twin_output.shape
)

display(
    smart_twin_output.head(10)
)


Complete Smart Twin output shape: (1930, 17)


,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Wear_Level,Health_Index,Anomaly_Score,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
0,CNC_001,CNC_Mill,1,68.987,0.2351,2.117,0.0000,100.000,0.1342,0,0.0001,LOW,257.17,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
1,CNC_001,CNC_Mill,2,69.431,0.1894,2.126,0.0000,99.961,0.2990,0,0.0001,LOW,258.95,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
2,CNC_001,CNC_Mill,3,68.779,0.2385,2.119,0.0022,99.760,0.2269,0,0.0001,LOW,259.01,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
3,CNC_001,CNC_Mill,4,68.926,0.2047,2.110,0.0000,99.884,0.0469,0,0.0001,LOW,256.91,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
4,CNC_001,CNC_Mill,5,68.704,0.2086,2.079,0.0006,99.802,0.2841,0,0.0001,LOW,260.68,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
5,CNC_001,CNC_Mill,6,68.667,0.2414,2.112,0.0000,99.807,0.1565,0,0.0001,LOW,254.38,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
6,CNC_001,CNC_Mill,7,68.250,0.2020,2.094,0.0149,98.654,0.3396,0,0.0001,LOW,251.90,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
7,CNC_001,CNC_Mill,8,69.082,0.1874,2.076,0.0138,98.693,0.2179,0,0.0001,LOW,251.83,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
8,CNC_001,CNC_Mill,9,68.661,0.1945,2.136,0.0037,99.414,0.2230,0,0.0001,LOW,251.49,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998
9,CNC_001,CNC_Mill,10,70.590,0.2267,2.136,0.0202,98.136,0.3307,0,0.0001,LOW,250.00,HEALTHY,HEALTHY,NORMAL OPERATION,99.989998


In [12]:
# ============================================
# STEP 12: HEALTH & RISK DISTRIBUTIONS
# ============================================

print("Overall Machine Health Distribution:")
display(
    smart_twin_output[
        "overall_status"
    ].value_counts()
)

print("\nFailure Risk Distribution:")
display(
    smart_twin_output[
        "failure_risk"
    ].value_counts()
)

print("\nRUL Status Distribution:")
display(
    smart_twin_output[
        "rul_status"
    ].value_counts()
)

print("\nMaintenance Action Distribution:")
display(
    smart_twin_output[
        "maintenance_action"
    ].value_counts()
)


Overall Machine Health Distribution:


overall_status
HEALTHY     1580
WARNING      206
CRITICAL     144
Name: count, dtype: int64


Failure Risk Distribution:


failure_risk
LOW       1806
HIGH       120
MEDIUM       4
Name: count, dtype: int64


RUL Status Distribution:


rul_status
HEALTHY     1580
WARNING      209
CRITICAL     141
Name: count, dtype: int64


Maintenance Action Distribution:


maintenance_action
NORMAL OPERATION         1580
SCHEDULE INSPECTION       206
IMMEDIATE MAINTENANCE     144
Name: count, dtype: int64

In [13]:
# ============================================
# STEP 13: CRITICAL MACHINES
# ============================================

critical_machines = smart_twin_output[
    smart_twin_output["overall_status"] == "CRITICAL"
].copy()

critical_machines = critical_machines.sort_values(
    by=[
        "failure_probability",
        "predicted_rul_cycles"
    ],
    ascending=[
        False,
        True
    ]
)

print(
    "Number of critical machine states:",
    len(critical_machines)
)

display(
    critical_machines.head(20)
)


Number of critical machine states: 144


,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Wear_Level,Health_Index,Anomaly_Score,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
568,PUMP_002,Hydraulic_Pump,309,92.551,0.8625,2.470,0.9852,16.142,1.9283,1,0.9983,HIGH,2.28,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.01
1094,MOTOR_004,Electric_Motor,235,83.603,0.8512,2.525,0.9734,17.206,1.9187,1,0.9981,HIGH,4.60,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.95
1695,PRESS_006,Hydraulic_Press,266,81.838,0.8775,2.947,0.9518,18.940,1.9780,1,0.9981,HIGH,7.80,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,3.24
1098,MOTOR_004,Electric_Motor,239,85.179,0.8878,2.541,0.9841,16.236,2.0298,1,0.9980,HIGH,1.68,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.79
1097,MOTOR_004,Electric_Motor,238,85.509,0.8497,2.592,1.0000,15.084,1.9814,1,0.9980,HIGH,2.10,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.96
1096,MOTOR_004,Electric_Motor,237,84.797,0.8508,2.573,0.9881,16.021,1.9413,1,0.9980,HIGH,2.33,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.05
1095,MOTOR_004,Electric_Motor,236,85.501,0.8595,2.552,0.9874,16.116,1.8770,1,0.9980,HIGH,3.30,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.44
1093,MOTOR_004,Electric_Motor,234,84.945,0.8449,2.554,0.9755,17.088,1.9032,1,0.9980,HIGH,4.55,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.94
1704,PRESS_006,Hydraulic_Press,275,80.552,0.8708,2.963,1.0000,15.000,1.9970,1,0.9979,HIGH,1.60,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.77
1920,ROBOT_007,Industrial_Robot,216,78.979,1.0244,2.418,0.9418,19.766,1.9433,1,0.9979,HIGH,6.12,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,2.58


In [14]:
# ============================================
# STEP 14: MACHINES REQUIRING MAINTENANCE
# ============================================

maintenance_required = smart_twin_output[
    smart_twin_output[
        "maintenance_action"
    ] != "NORMAL OPERATION"
].copy()

maintenance_required = maintenance_required.sort_values(
    by=[
        "predicted_rul_cycles",
        "failure_probability"
    ],
    ascending=[
        True,
        False
    ]
)

print(
    "Machine states requiring maintenance:",
    len(maintenance_required)
)

display(
    maintenance_required.head(20)
)


Machine states requiring maintenance: 350


,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Wear_Level,Health_Index,Anomaly_Score,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
1099,MOTOR_004,Electric_Motor,240,85.307,0.8174,2.492,1.0000,15.000,1.9690,1,0.9973,HIGH,1.43,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.74
858,COMP_003,Compressor,289,79.926,1.3850,2.401,0.9843,16.215,2.0085,1,0.9972,HIGH,1.49,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.77
1704,PRESS_006,Hydraulic_Press,275,80.552,0.8708,2.963,1.0000,15.000,1.9970,1,0.9979,HIGH,1.60,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.77
1929,ROBOT_007,Industrial_Robot,225,81.170,1.1698,2.403,1.0000,15.000,2.0478,1,0.9951,HIGH,1.66,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.96
1098,MOTOR_004,Electric_Motor,239,85.179,0.8878,2.541,0.9841,16.236,2.0298,1,0.9980,HIGH,1.68,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.79
1429,TURB_005,Turbine,330,79.028,1.2133,2.313,1.0000,15.000,2.0040,1,0.9964,HIGH,1.81,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.94
1703,PRESS_006,Hydraulic_Press,274,82.678,0.8516,2.978,0.9815,16.425,2.0333,1,0.9943,HIGH,1.86,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.08
258,CNC_001,CNC_Mill,259,82.636,1.2606,2.390,0.9969,15.269,2.0139,1,0.9965,HIGH,1.89,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.97
257,CNC_001,CNC_Mill,258,80.705,1.1958,2.397,0.9829,16.359,2.0787,1,0.9960,HIGH,1.91,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.00
569,PUMP_002,Hydraulic_Pump,310,93.677,0.9070,2.457,0.9862,16.032,2.0134,1,0.9977,HIGH,2.09,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.97


In [15]:
# ============================================
# STEP 15: TOP HEALTHIEST & LOWEST HEALTH
# ============================================

print("Lowest Health Scores:")
display(
    smart_twin_output.sort_values(
        "health_score",
        ascending=True
    ).head(20)
)

print("Highest Health Scores:")
display(
    smart_twin_output.sort_values(
        "health_score",
        ascending=False
    ).head(20)
)


Lowest Health Scores:


,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Wear_Level,Health_Index,Anomaly_Score,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
1099,MOTOR_004,Electric_Motor,240,85.307,0.8174,2.492,1.0000,15.000,1.9690,1,0.9973,HIGH,1.43,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.74
858,COMP_003,Compressor,289,79.926,1.3850,2.401,0.9843,16.215,2.0085,1,0.9972,HIGH,1.49,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.77
1704,PRESS_006,Hydraulic_Press,275,80.552,0.8708,2.963,1.0000,15.000,1.9970,1,0.9979,HIGH,1.60,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.77
1098,MOTOR_004,Electric_Motor,239,85.179,0.8878,2.541,0.9841,16.236,2.0298,1,0.9980,HIGH,1.68,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.79
1429,TURB_005,Turbine,330,79.028,1.2133,2.313,1.0000,15.000,2.0040,1,0.9964,HIGH,1.81,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.94
1097,MOTOR_004,Electric_Motor,238,85.509,0.8497,2.592,1.0000,15.084,1.9814,1,0.9980,HIGH,2.10,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.96
1929,ROBOT_007,Industrial_Robot,225,81.170,1.1698,2.403,1.0000,15.000,2.0478,1,0.9951,HIGH,1.66,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.96
569,PUMP_002,Hydraulic_Pump,310,93.677,0.9070,2.457,0.9862,16.032,2.0134,1,0.9977,HIGH,2.09,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.97
258,CNC_001,CNC_Mill,259,82.636,1.2606,2.390,0.9969,15.269,2.0139,1,0.9965,HIGH,1.89,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,0.97
257,CNC_001,CNC_Mill,258,80.705,1.1958,2.397,0.9829,16.359,2.0787,1,0.9960,HIGH,1.91,CRITICAL,CRITICAL,IMMEDIATE MAINTENANCE,1.00


Highest Health Scores:


,Machine_ID,Machine_Type,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Wear_Level,Health_Index,Anomaly_Score,failure_prediction,failure_probability,failure_risk,predicted_rul_cycles,rul_status,overall_status,maintenance_action,health_score
135,CNC_001,CNC_Mill,136,73.350,0.5091,2.144,0.3740,66.734,0.7797,0,0.0001,LOW,111.42,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
82,CNC_001,CNC_Mill,83,70.431,0.3374,2.103,0.1345,86.747,0.4828,0,0.0001,LOW,177.27,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
1531,PRESS_006,Hydraulic_Press,102,70.227,0.3305,2.304,0.1836,82.546,0.5874,0,0.0001,LOW,173.61,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
1309,TURB_005,Turbine,210,73.818,0.6228,2.152,0.4953,56.502,1.0828,0,0.0001,LOW,110.35,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
137,CNC_001,CNC_Mill,138,73.592,0.4779,2.167,0.3239,70.421,0.9211,0,0.0001,LOW,122.54,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
110,CNC_001,CNC_Mill,111,72.670,0.3913,2.164,0.2343,78.181,0.6076,0,0.0001,LOW,147.88,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
109,CNC_001,CNC_Mill,110,72.625,0.3316,2.170,0.2079,80.197,0.5911,0,0.0001,LOW,154.50,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
107,CNC_001,CNC_Mill,108,73.173,0.4555,2.154,0.2261,78.913,0.6738,0,0.0001,LOW,149.09,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
1820,ROBOT_007,Industrial_Robot,116,72.204,0.4809,2.282,0.3411,69.280,0.8332,0,0.0001,LOW,108.74,HEALTHY,HEALTHY,NORMAL OPERATION,100.0
1822,ROBOT_007,Industrial_Robot,118,73.381,0.5119,2.234,0.3390,69.355,0.7353,0,0.0001,LOW,107.21,HEALTHY,HEALTHY,NORMAL OPERATION,100.0


In [16]:
# ============================================
# STEP 16: SAVE SMART TWIN OUTPUT
# ============================================

output_path = "../data/processed/smart_twin_predictions.csv"

os.makedirs(
    "../data/processed",
    exist_ok=True
)

smart_twin_output.to_csv(
    output_path,
    index=False
)

print("Smart Twin output saved successfully!")
print("Path:", output_path)


Smart Twin output saved successfully!
Path: ../data/processed/smart_twin_predictions.csv


## Final Smart Twin Pipeline

The completed architecture is:

```text
Machine Sensor Data
        |
        +----------------------+
        |                      |
        v                      v
 XGBoost Classifier      LightGBM Regressor
        |                      |
        v                      v
Failure Probability       Predicted RUL
        |                      |
        +----------+-----------+
                   |
                   v
             Smart Twin Logic
                   |
        +----------+----------+
        |          |          |
        v          v          v
   Failure Risk  RUL Status  Health Score
        |          |          |
        +----------+----------+
                   |
                   v
        Overall Machine Status
                   |
                   v
       Maintenance Recommendation
```

The next project layer is the **Dashboard/API**, which will consume `smart_twin_output` rather than retraining either model.
